# Memory Model Validation

## Setup

In [1]:
import importlib
import logging
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set()
sys.path.append("..")

build_models = importlib.import_module("build-models")
appliance_model = "5150"

In [2]:
root_dir = pathlib.Path("..")
data_dir = root_dir / "data" / appliance_model / "8.2" / "default"

## Data: Basic Information
This data frame contains `top` snapshot-level info for each of the 5150 runs used. The primary key is `(num_streams, dedup_ratio, snapshot_index)`.

In [3]:
memory_data = build_models._get_memory_data(data_dir)
memory_data.head()

,snapshot_index,spoold_mem_fraction,spad_mem_fraction,total_mem,free_mem,used_mem,buff_cache_mem,total_swap,free_swap,used_swap,avail_mem,num_streams,dedup_ratio,run_timestamp
0,1,0.014,0.001,65199836,55609980,8046316,1543540,67108860,67108860,0,52893664,10,0.98,2019-10-05 12:00:14
1,2,0.014,0.001,65199836,55566060,8088516,1545260,67108860,67108860,0,52850536,10,0.98,2019-10-05 12:00:14
2,3,0.014,0.001,65199836,55162352,8407772,1629712,67108860,67108860,0,52488960,10,0.98,2019-10-05 12:00:14
3,4,0.016,0.001,65199836,54807548,8738256,1654032,67108860,67108860,0,52141004,10,0.98,2019-10-05 12:00:14
4,5,0.017,0.001,65199836,54350620,9071792,1777424,67108860,67108860,0,51708956,10,0.98,2019-10-05 12:00:14


In [4]:
memory_data.shape

(6221, 14)

This is the number of snapshots we have for each `num_streams`-`dedup_ratio` combination.

In [5]:
memory_data.groupby(["num_streams", "dedup_ratio"])["snapshot_index"].max()

num_streams  dedup_ratio
1            0.00            118
             0.50             58
             0.80             30
             0.98             16
2            0.00            237
             0.50            121
             0.80             56
             0.98             18
4            0.00            542
             0.50            278
             0.80            121
             0.98             34
8            0.00           1157
             0.50            585
             0.80            244
             0.98             48
10           0.00           1444
             0.50            744
             0.80            311
             0.98             59
Name: snapshot_index, dtype: int64

## Data: Testing Assumptions
There are two competing possible target values for the memory model. Neither includes the overhead.

The difference is whether the `buff/cache` part of the `top` output is included or not:

In [6]:
fixed_overhead = build_models._memory_overhead(appliance_model, memory_data)

memory_data["y1"] = (memory_data["used_mem"] + memory_data["buff_cache_mem"] 
    - (memory_data["spad_mem_fraction"] * memory_data["total_mem"])
    - (memory_data["spoold_mem_fraction"] * memory_data["total_mem"])
    - fixed_overhead)

memory_data["y2"] = memory_data["y1"] - memory_data["buff_cache_mem"]

Both are always greater than 0:

In [7]:
(memory_data["y1"] > 0).all() and (memory_data["y2"] > 0).all()

True

We expect the per-stream y-values to be independent of `num_streams` and `dedup_ratio`:

In [8]:
memory_data["ps1"] = memory_data["y1"] / memory_data["num_streams"]
memory_data["ps2"] = memory_data["y2"] / memory_data["num_streams"]

With `buff/cache`:

In [9]:
(memory_data.groupby(["num_streams", "dedup_ratio"])["ps1"]).max().unstack()

dedup_ratio,0.00,0.50,0.80,0.98
num_streams,,,,
1,6.740721e+06,5.185957e+06,4.451073e+06,4.217861e+06
2,6.703059e+06,4.451661e+06,3.145097e+06,2.391953e+06
4,8.492892e+06,4.662251e+06,2.608306e+06,1.403124e+06
8,6.870892e+06,6.850048e+06,3.244110e+06,8.693932e+05
10,5.495825e+06,5.492533e+06,3.600885e+06,7.333610e+05


Without:

In [10]:
(memory_data.groupby(["num_streams", "dedup_ratio"])["ps2"]).max().unstack()

dedup_ratio,0.00,0.50,0.80,0.98
num_streams,,,,
1,413337.2800,565525.2800,658057.2800,818325.2800
2,662978.6400,629478.6400,625828.6400,609498.6400
4,469460.3610,444588.4020,442978.3200,401567.3200
8,288701.6805,282575.1805,259057.2010,251838.7010
10,259104.1608,260436.9444,230592.5608,228487.3608


We decided to use the second definition, and simply return a constant value of 300,000 per stream. This is because the per-stream memory is roughly constant across dedup values, oversizing is not a huge problem and we mostly care about larger stream counts to begin with.